<a href="https://colab.research.google.com/github/yousefmohamed662006-debug/flyrank-ml-internship-youssef/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yousefmohamed662006-debug/flyrank-ml-internship-youssef/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My chosen lane is Refresh / Content Opportunity, and I frame it primarily
as a ranking task.

The decision is which content pages a content editor should review first.
The system would rank eligible pages from the highest to the lowest review
priority using their observed search, traffic, engagement, position, and
freshness signals.

This is a ranking task rather than only a classification task because the
team has limited review capacity and needs an ordered queue of pages, not
only a yes-or-no prediction. The output is decision support and does not
claim that updating a page will guarantee recovery.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
lane = "Refresh / Content Opportunity"
task_type = "Ranking"

decision = "Which content pages should be reviewed first?"
actor = "Content editor"
action = "Review the highest-ranked pages before lower-priority pages"

print("Lane:", lane)
print("ML task type:", task_type)
print("Decision:", decision)
print("Actor:", actor)
print("Supported action:", action)

Lane: Refresh / Content Opportunity
ML task type: Ranking
Decision: Which content pages should be reviewed first?
Actor: Content editor
Supported action: Review the highest-ranked pages before lower-priority pages


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The ideal target would be a future observed outcome: whether a content page
declines during a later time window after the features have been measured.

For this starter-data framing exercise, I will use `is_declining_label` as
a proxy. It equals 1 when `trend_direction` is `"down"` and 0 otherwise.
The direction is based on comparing the most recent 30-day impressions with
the previous 30-day impressions.

This is a beginner proxy rather than an ideal future target because it is
defined from the current snapshot. I will describe the results only as
directional decision support, not as proof that a page will decline or that
an update will cause recovery.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
target_name = "is_declining_label"
target_type = "Binary proxy"
positive_definition = '1 when trend_direction == "down", otherwise 0'
ideal_future_target = "Decline during a later outcome window"

print("Target/proxy:", target_name)
print("Type:", target_type)
print("Current proxy definition:", positive_definition)
print("Stronger future target:", ideal_future_target)

Target/proxy: is_declining_label
Type: Binary proxy
Current proxy definition: 1 when trend_direction == "down", otherwise 0
Stronger future target: Decline during a later outcome window


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My primary success metric is Precision@50.

It measures the proportion of the top 50 ranked pages that have a positive
`is_declining_label` proxy. This metric matches the real decision because a
content team has limited capacity and will review only the pages near the
top of the queue.

I will define a useful initial result as Precision@50 of at least 0.50. This
would mean that at least 25 of the first 50 reviewed pages match the decline
proxy. The metric supports prioritization, but it does not prove that
refreshing those pages will cause a performance recovery.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
metric = "Precision@50"
good_threshold = 0.50
review_capacity = 50

minimum_useful_pages = int(good_threshold * review_capacity)

print("Success metric:", metric)
print("Good threshold:", good_threshold)
print(
    f"A good result means at least {minimum_useful_pages} "
    f"of the top {review_capacity} pages match the proxy."
)

Success metric: Precision@50
Good threshold: 0.5
A good result means at least 25 of the top 50 pages match the proxy.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is one anonymized content page. Each row represents
one content item belonging to a pseudonymized client, together with its
content properties and aggregated search and engagement measurements.

The starter dataset is already the slice for my chosen Refresh / Content
Opportunity lane, so I will use its content-page rows directly.

For this framing exercise, I sketch `is_declining_label` as a binary proxy:
1 when `trend_direction` is `"down"` and 0 otherwise. This proxy is derived
from the current 30-day comparison and is not an independently observed
future outcome. Therefore, it should be used carefully as directional
decision support, and `trend_direction` or `trend_pct` must not later be
used as model features.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# Load the anonymized starter dataset directly from the public starter repo.
data_url = (
    "https://raw.githubusercontent.com/"
    "flyrank-bih/flyrank-ml-internship-starter/"
    "main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(data_url)

# The starter dataset already represents the Content Refresh lane.
lane_df = df.copy()

# Sketch the binary proxy target discussed in Section 2.
lane_df["is_declining_label"] = (
    lane_df["trend_direction"].eq("down").astype(int)
)

# Show only useful, anonymized columns for understanding one row.
display_columns = [
    "content_id",
    "client_id",
    "content_type",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "trend_direction",
    "is_declining_label",
]

print(f"Dataset shape: {lane_df.shape[0]:,} rows × {lane_df.shape[1]} columns")
print("Unit of analysis: one row = one anonymized content page")
print("\nProxy target distribution:")
print(lane_df["is_declining_label"].value_counts().sort_index())

lane_df[display_columns].head(10)

Dataset shape: 30,000 rows × 45 columns
Unit of analysis: one row = one anonymized content page

Proxy target distribution:
is_declining_label
0    13738
1    16262
Name: count, dtype: int64


,content_id,client_id,content_type,content_age_days,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,187,20,3803,29,0.76,10.6,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,25,15320,7,0.05,20.3,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,20,12581,11,0.09,36.5,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,463,22,11751,58,0.49,6.2,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,14,19140,24,0.13,44.0,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,147,20,3970,1,0.03,8.5,down,1
6,content_9a34b442b552,client_8722616204,keyword article,90,20,20,0,0.00,7.0,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,445,22,1724,1,0.06,21.2,stable,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,90,20,32574,29,0.09,46.0,down,1
9,content_c27558df2b0c,client_19581e27de,keyword article,257,104,1240,2,0.16,4.9,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A simple fixed rule might flag a page when its CTR is below the dataset
median while its impressions are above the median. This is a reasonable
baseline because it identifies pages with visibility but relatively weak
click performance.

However, one rule cannot represent all content-decline patterns. A page may
decline because of its search position, content age, time since its last
update, engagement, or a combination of several signals. Low CTR may also
be normal for a page with a low search position, while some declining pages
may still have an acceptable CTR.

A ranking model could combine these signals and learn different weights and
interactions instead of applying the same threshold to every page. Its output
would support a content editor in choosing which pages to inspect first.
This does not prove that ML will outperform the rule; that must be tested
honestly against a future outcome and a baseline.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build one simple fixed-rule baseline.
median_ctr = lane_df["ctr"].median()
median_impressions = lane_df["impressions_90d"].median()

lane_df["fixed_rule_flag"] = (
    (lane_df["ctr"] < median_ctr)
    & (lane_df["impressions_90d"] > median_impressions)
).astype(int)

# Compare the simple rule with the current decline proxy.
comparison = pd.crosstab(
    lane_df["fixed_rule_flag"],
    lane_df["is_declining_label"],
    rownames=["Fixed rule"],
    colnames=["Decline proxy"],
    margins=True,
)

true_positive = (
    (lane_df["fixed_rule_flag"] == 1)
    & (lane_df["is_declining_label"] == 1)
).sum()

false_positive = (
    (lane_df["fixed_rule_flag"] == 1)
    & (lane_df["is_declining_label"] == 0)
).sum()

false_negative = (
    (lane_df["fixed_rule_flag"] == 0)
    & (lane_df["is_declining_label"] == 1)
).sum()

rule_precision = (
    true_positive / (true_positive + false_positive)
    if true_positive + false_positive > 0
    else 0
)

rule_recall = (
    true_positive / (true_positive + false_negative)
    if true_positive + false_negative > 0
    else 0
)

print(f"Median CTR threshold: {median_ctr:.2f}")
print(f"Median impressions threshold: {median_impressions:,.0f}")
print(f"Pages flagged by the fixed rule: {lane_df['fixed_rule_flag'].sum():,}")
print(f"Rule precision against the proxy: {rule_precision:.3f}")
print(f"Rule recall against the proxy: {rule_recall:.3f}")

comparison

Median CTR threshold: 0.07
Median impressions threshold: 731
Pages flagged by the fixed rule: 3,309
Rule precision against the proxy: 0.675
Rule recall against the proxy: 0.137


Decline proxy,0,1,All
Fixed rule,,,
0,12663,14028,26691
1,1075,2234,3309
All,13738,16262,30000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.


- [x] Every section is filled with markdown reasoning and supporting code.
- [x] I framed one clear ML task: ranking content pages by review priority.
- [x] I defined `is_declining_label` as a binary proxy, not a future observed outcome.
- [x] I selected Precision@50 as the primary success metric and defined 0.50 as an initial useful threshold.
- [x] I loaded the real starter dataset and showed the unit of analysis as a dataframe.
- [x] One row represents one anonymized content page.
- [x] I compared the framing with a simple fixed-rule baseline.
- [x] I explained that the baseline is selective but misses many declining pages.
- [x] I tied the output to a real action: helping a content editor choose which pages to inspect first.
- [x] I used careful language such as proxy, observed, directional, and decision support.
- [x] I included no client names, real URLs, or private search queries.
- [x] The notebook runs from top to bottom without errors.